# HttpClient
This notebook demonstrates the public `HttpClient.request(...)` escape hatch.
It uses a fake `requests.Session` so it can run without network access (e.g. in CI).

**Note:** `HttpClient` can be used, but it's not advised to do so, as is lacks some convenience mechanics (like directly setting up endpoints etc.).

#### Set up a fake `requests` session

This is only relevant for the purpose to make the notebook runnable without having access to any online service.

In [1]:
from notebooks.fake_http_session import FakeResponse, FakeSession

fake = FakeSession(
    response=FakeResponse(
        url="https://example.org/api/v1/endpoints",
        status_code=200,
        text="['fake response content']",
    )
)

## Using `HttpClient`

In [2]:
from courier.base_client import HttpClient
from courier.transport.url import join_url

`HttpClient` can take existing `requests.Session()` instances. We use this here to use a fake session.
`HttpClient` automatically sets up authorization via a JSON-webtoken, if `token` is passed.

In [3]:
token = "<TOKEN>"
address = "example.org"
client = HttpClient(address, token=token, session=fake) # when address and toke correspond to a real existing service, no value for session has to be given.

In [4]:
client.base_url

'https://example.org'

#### defining an endpoint

Endpoint addresses are passed as urls to `HttpClient`. Using courier.transport.url.join_url ensures a sanitized url (no double //, no whitespaces etc).

In [5]:
# url = client.base_url + "/api/v1/endpoints"
url = join_url(client.base_url, segments=["api", "v1", "endpoints"])

### make a request

You can specify the method (`GET`, `PUT`, `POST` etc.)

In [6]:
resp = client.request(method="GET", url=url)

In [7]:
resp.status_code

200

In [8]:
resp.text

"['fake response content']"